# 04. BM25 & Oracle Context RAG Evaluation Suite
Evaluates BM25 Top-1, Top-3, Top-5, Recall@1, Recall@3, Recall@5, MRR, Oracle Gold-Context RAG, and Generation Quality conditional on Retrieval Hit vs Retrieval Miss across 14,576 Drug Formulary passages.

In [1]:
# Cell 1: Thư viện & Environment
!pip install -q evaluate bert_score bitsandbytes accelerate transformers rank_bm25 scipy
import os, sys, json, time, torch, numpy as np, pandas as pd
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate
print('✅ PyTorch Version:', torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 49.2 MB/s eta 0:00:00
✅ PyTorch Version: 2.10.0+cu128


In [2]:
# Cell 2: Dataset Search & Resolution
possible_paths = [
    '/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_root = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower() or 'vnese' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

if data_path is None:
    raise FileNotFoundError("Dataset file not found!")

print('✅ Resolved Dataset Path:', data_path)
with open(data_path, 'r', encoding='utf-8') as f: full_dataset = json.load(f)
test_data = full_dataset[-500:]
print('Total dataset size:', len(full_dataset), '| Test size:', len(test_data))
assert len(test_data) == 500, 'Test size must be 500'


✅ Resolved Dataset Path: /kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json
Total dataset size: 14700 | Test size: 500


In [3]:
# Cell 3: Build Passage Corpus & BM25 Index over ALL 14,576 unique passages (k1=1.5, b=0.75)
passages = []
seen_texts = set()
for item in full_dataset:
    text = item.get('right_answer', item.get('positive_answer'))
    if text and text not in seen_texts:
        seen_texts.add(text)
        passages.append({
            'passage_id': f'PASSAGE-{len(passages)+1:05d}',
            'text': text,
            'category': item.get('category', 'general')
        })

tokenized_corpus = [p['text'].lower().split() for p in passages]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print(f'✅ Built BM25 Index over {len(passages)} unique Drug Formulary passages (k1=1.5, b=0.75)!')


✅ Built BM25 Index over 14576 unique Drug Formulary passages (k1=1.5, b=0.75)!


In [4]:
# Cell 4: Evaluate Retrieval Metrics: Recall@1, Recall@3, Recall@5, and MRR
rag_records = []
recalls_1, recalls_3, recalls_5, mrrs = [], [], [], []

for q_idx, item in enumerate(tqdm(test_data, desc='Evaluating Retrieval Metrics')):
    q_text = item['question']
    gold_text = item.get('right_answer', item.get('positive_answer'))
    q_toks = q_text.lower().split()
    
    t0 = time.time()
    scores = bm25.get_scores(q_toks)
    ret_lat = time.time() - t0
    
    top5_idx = np.argsort(scores)[::-1][:5]
    ret_p_ids = [passages[i]['passage_id'] for i in top5_idx]
    ret_scores = [float(scores[i]) for i in top5_idx]
    
    # Gold passage match
    r1 = 1 if passages[top5_idx[0]]['text'] == gold_text else 0
    r3 = 1 if any(passages[i]['text'] == gold_text for i in top5_idx[:3]) else 0
    r5 = 1 if any(passages[i]['text'] == gold_text for i in top5_idx[:5]) else 0
    
    rank = 0
    for rank_idx, idx_val in enumerate(top5_idx, 1):
        if passages[idx_val]['text'] == gold_text:
            rank = rank_idx; break
    mrr = 1.0 / rank if rank > 0 else 0.0
    
    recalls_1.append(r1)
    recalls_3.append(r3)
    recalls_5.append(r5)
    mrrs.append(mrr)
    
    rag_records.append({
        'question_id': f'Q-{q_idx:03d}',
        'category': item.get('category', 'general'),
        'question': q_text,
        'gold_reference': gold_text,
        'retrieved_passage_ids': ret_p_ids,
        'retrieval_scores': ret_scores,
        'gold_passage_id': f'PASSAGE-{q_idx+1:05d}',
        'retrieval_hit': r1,
        'retrieval_hit_top3': r3,
        'retrieval_hit_top5': r5,
        'mrr': mrr,
        'retrieval_latency': ret_lat
    })

print('========================================================================')
print(f'✅ BM25 Recall@1: {np.mean(recalls_1)*100:.2f}% ({sum(recalls_1)}/500)')
print(f'✅ BM25 Recall@3: {np.mean(recalls_3)*100:.2f}% ({sum(recalls_3)}/500)')
print(f'✅ BM25 Recall@5: {np.mean(recalls_5)*100:.2f}% ({sum(recalls_5)}/500)')
print(f'✅ BM25 MRR:      {np.mean(mrrs):.4f}')
print('========================================================================')


Evaluating Retrieval Metrics: 100%|██████████| 500/500 [00:24<00:00, 20.70it/s]

✅ BM25 Recall@1: 19.20% (96/500)
✅ BM25 Recall@3: 28.40% (142/500)
✅ BM25 Recall@5: 32.80% (164/500)
✅ BM25 MRR:      0.2433


In [5]:
# Cell 5: Qwen2.5-7B Generation (BM25 Top-1 RAG vs Oracle Gold-Context RAG)
model_id = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
bertscore = evaluate.load('bertscore')

gen_records = []
for r in tqdm(rag_records, desc='Generating RAG Responses'):
    # BM25 Top-1 Context
    retrieved_text = passages[int(r['retrieved_passage_ids'][0].split('-')[1])-1]['text']
    prompt_bm25 = f"<|im_start|>user\nNgữ cảnh: {retrieved_text}\n\nCâu hỏi: {r['question']}<|im_end|>\n<|im_start|>assistant\n"
    inp_bm25 = tokenizer(prompt_bm25, return_tensors='pt').to(model.device)
    
    t0 = time.time()
    with torch.no_grad():
        out_bm25 = model.generate(**inp_bm25, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    lat_gen = time.time() - t0
    text_bm25 = tokenizer.decode(out_bm25[0][inp_bm25.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Oracle Gold Context
    prompt_oracle = f"<|im_start|>user\nNgữ cảnh: {r['gold_reference']}\n\nCâu hỏi: {r['question']}<|im_end|>\n<|im_start|>assistant\n"
    inp_oracle = tokenizer(prompt_oracle, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_oracle = model.generate(**inp_oracle, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    text_oracle = tokenizer.decode(out_oracle[0][inp_oracle.input_ids.shape[1]:], skip_special_tokens=True)
    
    r_dict = dict(r)
    r_dict['prompt_tokens'] = inp_bm25.input_ids.shape[1]
    r_dict['generation_latency'] = lat_gen
    r_dict['generated_text'] = text_bm25
    r_dict['oracle_generated_text'] = text_oracle
    gen_records.append(r_dict)

df_rag = pd.DataFrame(gen_records)
print('Computing BERTScore metrics for RAG...')
bs_p = bertscore.compute(predictions=df_rag['generated_text'].tolist(), references=df_rag['gold_reference'].tolist(), model_type='bert-base-multilingual-cased')['f1']
df_rag['BS_positive'] = bs_p
bs_o = bertscore.compute(predictions=df_rag['oracle_generated_text'].tolist(), references=df_rag['gold_reference'].tolist(), model_type='bert-base-multilingual-cased')['f1']
df_rag['BS_oracle'] = bs_o

os.makedirs('outputs', exist_ok=True)
os.makedirs('results', exist_ok=True)
df_rag.to_csv('outputs/rag_evaluation_outputs.csv', index=False)
with open('outputs/rag_evaluation_outputs.jsonl', 'w', encoding='utf-8') as f:
    for row in gen_records: f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('========================================================================')
print('✅ Saved outputs/rag_evaluation_outputs.csv and outputs/rag_evaluation_outputs.jsonl!')
print('========================================================================')
assert len(df_rag) == 500, 'RAG test size must be 500'
print('✅ Automated RAG Assertions Passed 100%!')


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Saved outputs/rag_evaluation_outputs.csv and outputs/rag_evaluation_outputs.jsonl!
✅ Automated RAG Assertions Passed 100%!
